# LangChain: What It's Hiding From You

LangChain is a framework for building LLM applications. You've already used it in your `workspace_memory` plugin — `PyPDFLoader`, `RecursiveCharacterTextSplitter`, `HuggingFaceEmbeddings`, ChromaDB. This notebook opens the hood. You'll see exactly what each LangChain abstraction is doing underneath. Then in notebook 05, we replace all of it with ~150 lines of pure Python and NumPy.

**4 exercises.** Each is 1–5 lines of code. The goal isn't volume — it's understanding every layer before it disappears.

### The abstraction stack
| Layer | LangChain | What's underneath |
|-------|-----------|-------------------|
| Loading | `TextLoader`, `PyPDFLoader` | `open(file).read()` |
| Splitting | `RecursiveCharacterTextSplitter` | String slicing with overlap |
| Embedding | `HuggingFaceEmbeddings` | `SentenceTransformer.encode()` |
| Storage | `Chroma.from_texts()` | NumPy array + dot-product search |
| Retrieval | `RetrievalQA` chain | Prompt construction + LLM call |

### Prerequisites
```bash
pip install langchain langchain-community langchain-text-splitters sentence-transformers chromadb pypdf
```
> **Note:** On first run, `sentence-transformers` downloads `all-MiniLM-L6-v2` (~80MB). This takes ~30 seconds once, then it's cached locally. The httpx SSL patch in the setup cell is needed for corporate VPN environments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── httpx SSL patch for corporate VPN / SSL-intercepting proxies ──────────────
# huggingface_hub uses httpx which has its own SSL stack — patch before any download.
import httpx

_orig_client = httpx.Client.__init__
def _patched_client(self, *args, **kwargs):
    kwargs['verify'] = False
    _orig_client(self, *args, **kwargs)
httpx.Client.__init__ = _patched_client

_orig_async = httpx.AsyncClient.__init__
def _patched_async(self, *args, **kwargs):
    kwargs['verify'] = False
    _orig_async(self, *args, **kwargs)
httpx.AsyncClient.__init__ = _patched_async

print('Setup complete. httpx SSL patch applied.')

---
## Part 1: What LangChain Is (and Isn't)

LangChain is a collection of abstractions that standardize five things: **document loading, text splitting, embedding, vector storage, and chain composition.** It's genuinely useful for wiring components together quickly, and for writing provider-agnostic code that works with OpenAI today and a local model tomorrow.

The cost: it hides the parts you need to understand.

Here's the full side-by-side:

| Component | What you wrote (LangChain) | What you'll write in notebook 05 (raw Python) |
|-----------|---------------------------|-----------------------------------------------|
| Loading | `TextLoader(path).load()` | `open(path).read()` |
| Splitting | `RecursiveCharacterTextSplitter(chunk_size=500).split_documents(docs)` | `word_boundary_chunk(text, 500, 50)` |
| Embedding | `HuggingFaceEmbeddings(model_name=...).embed_documents(texts)` | `SentenceTransformer(...).encode(texts)` |
| Storage | `Chroma.from_texts(texts, embedder)` | `np.array([model.encode(t) for t in texts])` |
| Retrieval | `vectorstore.similarity_search(query, k=3)` | `sorted(zip(texts, scores))[:3]` |

Each row is a learning exercise. Let's go through them.

---
## Part 2: Document Loaders

A document loader reads a file and returns a list of `Document` objects. Each `Document` has two attributes:
- `.page_content` — the text string
- `.metadata` — a dict with source path, page number, etc.

That's it. The entire abstraction is: read file → wrap in object.

In [ ]:
from langchain_community.document_loaders import TextLoader

# Create a sample file to load
with open('/tmp/sample.txt', 'w') as f:
    f.write('Gradient descent minimizes loss functions.\nThe learning rate controls step size.')

loader = TextLoader('/tmp/sample.txt')
docs = loader.load()

print(f'Type:     {type(docs[0])}')
print(f'Content:  {docs[0].page_content!r}')
print(f'Metadata: {docs[0].metadata}')

In [ ]:
# What TextLoader is actually doing:
with open('/tmp/sample.txt') as f:
    text = f.read()

print(repr(text))
print()
print("That's it. LangChain wraps this in a Document object with a .metadata dict.")
print('PyPDFLoader does the same — except it calls pypdf.PdfReader first.')

### ✏️ Exercise 1: Write your own `SimpleLoader`

Mimic `TextLoader` — read a file and return a list containing one dict with `page_content` and `metadata` keys. Your loader doesn't need to use a `Document` class; a plain dict is fine.

**2–3 lines.**

In [ ]:
def SimpleLoader(path: str) -> list:
    """
    Read a text file and return a list containing one dict:
    [{'page_content': <file text>, 'metadata': {'source': path}}]
    """
    # ✏️ YOUR TURN: open the file, read it, return the list
    raise NotImplementedError('Fill this in')

# ── Test ─────────────────────────────────────────────────────────────────────
result = SimpleLoader('/tmp/sample.txt')
assert isinstance(result, list), 'Should return a list'
assert len(result) == 1, 'Should contain exactly one item'
assert result[0]['page_content'] == docs[0].page_content, 'Content should match TextLoader'
assert result[0]['metadata']['source'] == '/tmp/sample.txt', 'Metadata should have source key'
print('✅ Passed! Your SimpleLoader produces the same content as TextLoader.')

<details>
<summary>💡 Solution (click to expand)</summary>

```python
def SimpleLoader(path: str) -> list:
    with open(path) as f:
        content = f.read()
    return [{'page_content': content, 'metadata': {'source': path}}]
```

The LangChain version adds encoding detection, error handling, and returns `Document` objects instead of dicts. The core logic is identical.

</details>

---
## Part 3: Text Splitters

The splitter takes a list of `Document` objects and returns a longer list of smaller `Document` objects.

`RecursiveCharacterTextSplitter` tries separators in order — `['\n\n', '\n', '.', ' ', '']` — splitting only when the current separator still produces chunks that are too large. This preserves structure: paragraph breaks first, then line breaks, then sentences, then words.

The key behavior: if a chunk fits within `chunk_size`, it is **not** split further, regardless of which separator matched. It only recurses when needed.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use a slightly longer sample to produce multiple chunks
with open('/tmp/sample.txt', 'w') as f:
    f.write(
        'Gradient descent is an optimization algorithm used to minimize a loss function.\n'
        'It works by iteratively moving in the direction of steepest descent.\n\n'
        'The learning rate controls how large a step we take each iteration.\n'
        'A learning rate that is too large causes the algorithm to overshoot.\n\n'
        'Stochastic gradient descent computes the gradient from one randomly chosen sample.\n'
        'This is much faster per update but introduces noise into each gradient estimate.'
    )

loader = TextLoader('/tmp/sample.txt')
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print(f'Input: 1 document | Output: {len(chunks)} chunks')
for i, chunk in enumerate(chunks[:4]):
    print(f'\n[{i}] {len(chunk.page_content)} chars: {chunk.page_content[:80]!r}...')

Here's roughly what's happening inside `RecursiveCharacterTextSplitter`:

In [ ]:
def simple_recursive_split(text: str, chunk_size: int, separators: list = None) -> list:
    """Simplified recursive splitter — tries separators in order."""
    if separators is None:
        separators = ['\n\n', '\n', ' ', '']

    sep = separators[0]
    parts = text.split(sep) if sep else list(text)

    chunks = []
    current = ''
    for part in parts:
        candidate = (current + sep + part).strip() if current else part
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            # If this single part is still too big, recurse with next separator
            if len(part) > chunk_size and len(separators) > 1:
                chunks.extend(simple_recursive_split(part, chunk_size, separators[1:]))
                current = ''
            else:
                current = part
    if current:
        chunks.append(current)
    return chunks

sample_text = docs[0].page_content
raw_chunks = simple_recursive_split(sample_text, chunk_size=200)
print(f'Raw splitter produced: {len(raw_chunks)} chunks')
for i, c in enumerate(raw_chunks):
    print(f'  [{i}] {len(c)} chars')

### ✏️ Exercise 2: What does `chunk_overlap=30` actually mean?

Overlap means the next chunk **re-reads** the last N characters of the previous chunk. This preserves context across boundaries — if a sentence starts near the end of chunk 0, it also appears at the start of chunk 1.

Given a chunk that ends at character index `current_end`, and an overlap of `overlap` characters, the **next chunk starts at** `current_end - overlap`.

Write a function that computes this. **1 line.**

In [ ]:
def next_chunk_start(current_end: int, overlap: int) -> int:
    """
    Return the start index of the next chunk given where the current one ended
    and how many characters to overlap.

    Example: current_end=200, overlap=30  →  next chunk starts at 170
    """
    # ✏️ YOUR TURN: one line
    raise NotImplementedError('Fill this in')

# ── Test ─────────────────────────────────────────────────────────────────────
assert next_chunk_start(200, 30) == 170, f'Got {next_chunk_start(200, 30)}, expected 170'
assert next_chunk_start(500, 0) == 500,  'Zero overlap: next chunk starts right after current'
assert next_chunk_start(100, 100) == 0,  'Full overlap: next chunk starts at beginning'
print('✅ Passed! Overlap = re-reading the last N characters of the previous chunk.')

<details>
<summary>💡 Solution (click to expand)</summary>

```python
def next_chunk_start(current_end: int, overlap: int) -> int:
    return current_end - overlap
```

So a chunk ending at character 200 with overlap=30 means the next chunk starts at character 170 — it re-reads characters 170–199 from the previous chunk. This is exactly what `RecursiveCharacterTextSplitter` does internally. In the notebook 05 `word_boundary_chunk` function, this is the `start = split_point + 1 - overlap` line.

</details>

---
## Part 4: Embeddings

LangChain's `HuggingFaceEmbeddings` is a wrapper around `sentence-transformers`. Under the hood it calls `SentenceTransformer.encode()` — exactly what you'll use directly in notebook 05.

Let's prove they're identical.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

# LangChain version — downloads model on first run
print('Loading via LangChain...')
lc_embedder = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
lc_result = lc_embedder.embed_documents(['Gradient descent minimizes loss.'])

# Raw version — same download, same model
print('Loading directly via sentence-transformers...')
raw_model = SentenceTransformer('all-MiniLM-L6-v2')
raw_result = raw_model.encode(['Gradient descent minimizes loss.'], convert_to_numpy=True)

print()
print(f'LangChain shape: {np.array(lc_result).shape}')
print(f'Raw shape:       {raw_result.shape}')
print(f'Max difference:  {np.max(np.abs(np.array(lc_result) - raw_result)):.2e}')
print('← They are identical. LangChain adds zero value here.')

The `HuggingFaceEmbeddings` class is roughly 80 lines of config wiring and provider abstraction on top of that one `.encode()` call. The abstraction is useful when you want to swap `all-MiniLM-L6-v2` for OpenAI embeddings without rewriting downstream code. But when you need to debug why retrieval is producing bad results, you don't want those 80 lines between you and the model.

---
## Part 5: Vector Stores (ChromaDB)

ChromaDB stores your embeddings on disk and provides similarity search. Under the hood it's doing the same dot-product search you'll implement in notebook 05, plus an approximate nearest-neighbor index (HNSW) for scale. For small collections (< 100k chunks) it's essentially NumPy with persistence.

Let's build a Chroma vector store from the chunks we already have.

In [ ]:
from langchain_community.vectorstores import Chroma

# Build a vector store from the chunk texts
texts = [chunk.page_content for chunk in chunks]
vectorstore = Chroma.from_texts(texts, lc_embedder)

# Query it directly
query = 'What controls the step size in gradient descent?'
results = vectorstore.similarity_search(query, k=3)

print(f'Query: "{query}"')
print(f'Returned {len(results)} results:\n')
for i, r in enumerate(results):
    print(f'[{i}] Type: {type(r).__name__}')
    print(f'     .page_content: {r.page_content[:80]!r}...')
    print(f'     .metadata:     {r.metadata}')

### ✏️ Exercise 3: What does `similarity_search` actually return?

The standard `similarity_search` gives you `Document` objects but **hides the scores**. There's a companion method `similarity_search_with_score` that exposes them.

Your task: call `similarity_search_with_score` and inspect the score type. Is it **similarity** (higher = more similar) or **distance** (lower = more similar)?

In [ ]:
# ✏️ YOUR TURN: call vectorstore.similarity_search_with_score(query, k=3)
# Then print each result's score and the first 60 chars of its content.
# Is the best result the highest score or the lowest?

# scored_results = ???
raise NotImplementedError('Call similarity_search_with_score here')

# ── Test ─────────────────────────────────────────────────────────────────────
assert isinstance(scored_results, list), 'Should be a list'
assert all(isinstance(item, tuple) for item in scored_results), 'Each item should be a tuple'
doc, score = scored_results[0]
assert isinstance(float(score), float), 'Score should be numeric'
print('\n✅ Passed!')

<details>
<summary>💡 Solution (click to expand)</summary>

```python
scored_results = vectorstore.similarity_search_with_score(query, k=3)

for doc, score in scored_results:
    print(f'Score: {score:.4f}  |  {doc.page_content[:60]}...')
```

**The gotcha:** ChromaDB's default metric is **L2 distance** (Euclidean), not cosine similarity. That means a **lower score = more similar** — the opposite of what you'd expect from a "similarity" method.

This trips people up constantly. When you build the raw version in notebook 05 with cosine similarity, you'll get a higher score for the best match and sort descending. With Chroma's default you sort ascending. The underlying math is different, which can cause the ranked order to differ slightly between the two implementations.

You can switch Chroma to cosine with `collection_metadata={'hnsw:space': 'cosine'}` — but only if you know to look for it.

</details>

---
## Part 6: The Full LangChain RAG Chain

Here's everything assembled into a `RetrievalQA` chain. This is what most tutorials show you. It works — but when it breaks, you have no idea which layer to look at.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# Note: we're not invoking a full LLM chain here to avoid API cost.
# The full chain would look like:
#
#   from langchain.chains import RetrievalQA
#   llm = ...  # LangChain-wrapped LLM (ChatOpenAI, ChatAnthropic, etc.)
#   qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
#   answer = qa_chain.invoke({'query': 'What is gradient descent?'})
#
# Instead, let's see what the retriever produces on its own:

retrieved = retriever.invoke('What is the learning rate?')
print(f'Retrieved {len(retrieved)} documents:\n')
for i, doc in enumerate(retrieved):
    print(f'[{i}] {doc.page_content[:100]}...')

---
## Part 7: The Cost of Abstraction

Here's the honest accounting:

| Component | What it adds | What it hides |
|-----------|-------------|---------------|
| Document loaders | Uniform interface across file types | It's just file reading + optional PDF parsing |
| Text splitters | Separator hierarchy, length checks | The overlap arithmetic |
| Embeddings | Provider switching (OpenAI ↔ HuggingFace ↔ Cohere) | It's just `.encode()` |
| ChromaDB | Persistence, HNSW index for scale | Distance vs. similarity confusion |
| Chains | Composability, streaming | Where errors come from |

For **learning**, the abstractions hurt you — they hide the parts that break and the parts you need to tune.

For **production at scale**, some are genuinely useful: ChromaDB's HNSW index matters at 1M+ chunks, provider-agnostic embeddings matter if you switch models. Know which you actually need before reaching for the framework.

In [ ]:
def draw_abstraction_stacks():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

    PALETTE = {
        'blue':   '#4C9BE8',
        'green':  '#5DBE7C',
        'orange': '#E8A040',
        'red':    '#E8704C',
        'purple': '#9B59B6',
    }

    lc_layers = [
        ('RetrievalQA chain',                PALETTE['purple']),
        ('Chroma vectorstore',               PALETTE['red']),
        ('HuggingFaceEmbeddings',            PALETTE['orange']),
        ('RecursiveCharacterTextSplitter',   PALETTE['green']),
        ('TextLoader / PyPDFLoader',         PALETTE['blue']),
    ]

    raw_layers = [
        ('Prompt string + LLM call',         PALETTE['purple']),
        ('np.dot() + sort (cosine search)',  PALETTE['red']),
        ('SentenceTransformer.encode()',      PALETTE['orange']),
        ('word_boundary_chunk()',            PALETTE['green']),
        ('open(file).read()',                PALETTE['blue']),
    ]

    for ax, layers, title in [
        (ax1, lc_layers,  'LangChain Stack'),
        (ax2, raw_layers, 'Raw Python Stack (notebook 05)'),
    ]:
        ax.set_xlim(0, 10)
        ax.set_ylim(-0.5, len(layers) * 1.4)
        ax.axis('off')
        ax.set_title(title, fontsize=12, fontweight='bold', pad=12)

        for i, (label, color) in enumerate(layers):
            y = i * 1.4
            rect = mpatches.FancyBboxPatch(
                (0.5, y), 9.0, 1.0,
                boxstyle='round,pad=0.08', lw=2,
                edgecolor=color, facecolor=color + '33'
            )
            ax.add_patch(rect)
            ax.text(5.0, y + 0.5, label,
                    ha='center', va='center',
                    fontsize=9.5, fontweight='bold', color=color)

            if i < len(layers) - 1:
                ax.annotate('', xy=(5.0, y + 1.0), xytext=(5.0, y + 1.35),
                            arrowprops=dict(arrowstyle='->', lw=1.5, color='#aaa'))

        ax.text(5.0, -0.35, '← document enters here',
                ha='center', fontsize=7.5, color='#888', style='italic')
        top_y = (len(layers) - 1) * 1.4 + 1.05
        ax.text(5.0, top_y, 'LLM answer exits here →',
                ha='center', fontsize=7.5, color='#888', style='italic')

    fig.suptitle('Same pipeline. Same operations. Different visibility.',
                 fontsize=11, color='#555', y=0.02)
    plt.tight_layout()
    plt.show()

draw_abstraction_stacks()

---
### ✏️ Exercise 4: Spot the Difference

Run both the LangChain retriever and the `MinimalVectorStore` below on the same query. Do they return the same top chunk? If not — why might they differ?

The `MinimalVectorStore` class is inlined here from the notebook 05 pattern. Read it — every line should make sense after what you've seen.

In [ ]:
# ── MinimalVectorStore: the notebook 05 raw implementation ───────────────────
class MinimalVectorStore:
    def __init__(self):
        self.texts = []
        self.embeddings = np.array([])

    def index(self, texts: list):
        self.texts = texts
        self.embeddings = raw_model.encode(texts, convert_to_numpy=True)

    def retrieve(self, query: str, k: int = 3):
        q = raw_model.encode([query], convert_to_numpy=True)[0]
        # Cosine similarity: dot product / (norm_a * norm_b)
        scores = np.dot(self.embeddings, q) / (
            np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(q)
        )
        ranked = sorted(zip(self.texts, scores.tolist()), key=lambda x: x[1], reverse=True)
        return ranked[:k]


# ── Build the raw store on the same texts ────────────────────────────────────
raw_store = MinimalVectorStore()
raw_store.index(texts)

compare_query = 'What controls the step size in gradient descent?'

# LangChain retriever result
lc_top = retriever.invoke(compare_query)
lc_top_text = lc_top[0].page_content if lc_top else '(no result)'

# ✏️ YOUR TURN: call raw_store.retrieve(compare_query, k=3) and store top result in raw_top_text
# Then print both side by side and note whether they agree.
raise NotImplementedError('Call raw_store.retrieve() and compare the top result')

print(f'Query: "{compare_query}"\n')
print(f'LangChain top:  {lc_top_text[:100]!r}...')
print(f'Raw store top:  {raw_top_text[:100]!r}...')
print()
print('Same?', lc_top_text[:60] == raw_top_text[:60])

<details>
<summary>💡 Solution (click to expand)</summary>

```python
raw_results = raw_store.retrieve(compare_query, k=3)
raw_top_text = raw_results[0][0]
```

**Why they might differ:**

- **Distance vs. similarity:** Chroma's default metric is L2 distance (lower = better). `MinimalVectorStore` uses cosine similarity (higher = better). These are mathematically different and can produce different rankings, especially for chunks that have similar cosine similarity but different vector magnitudes.

- **Normalization:** `all-MiniLM-L6-v2` outputs unit-normalized vectors, so L2 distance and cosine similarity rank identically for this model. But this is model-specific — it's not guaranteed.

**The takeaway:** when retrieval gives you bad answers, you need to know *which* similarity metric your store is using. LangChain hides this. Your raw implementation makes it explicit — it's right there in the `np.dot()` / `np.linalg.norm()` line.

</details>

---
## Looking Forward

You now know exactly what LangChain is doing at each layer:

- **Loaders** → `open().read()`
- **Splitters** → string slicing with overlap arithmetic
- **Embeddings** → `SentenceTransformer.encode()`
- **ChromaDB** → dot-product search (with an HNSW index bolted on for scale)
- **Chains** → prompt string construction + an LLM call

In **notebook 05**, you'll replace every one of these with raw Python. The code will be shorter, faster to debug, and you'll understand every line — because you just built each piece by hand.

The LangChain abstractions aren't wrong. They're just not where you should live while you're learning.

---
*Built for [ML Edge](https://mle-edge.dev) — self-directed ML curriculum*